# 15.8 Heaps and Priority Queues

**Prerequisites:** 15.7 Trees, 15.1 Complexity Analysis, 15.2 Python's Built-ins  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The heap property, and why a heap is **not** a search tree
- 🔴 A tree stored in a **flat array** - the index arithmetic that makes it work
- `heapq`: push, pop, peek - and that it is **min-only**
- 🔴 **`heapify` is O(n), not O(n log n)** - counter-intuitive, and provable
- Top-k in O(n log k) instead of O(n log n)
- Priority queues, the tie-breaking trap, and the counter fix
- Merging k sorted sequences
- **Running median** with two heaps
- Interview questions, worked

---

## What a heap is for

One question, asked repeatedly: **"what is the smallest thing right now?"** — while items keep arriving and leaving.

| Approach | Insert | Get minimum | Remove minimum |
|---|---|---|---|
| Unsorted list | O(1) | O(n) | O(n) |
| Sorted list | O(n) (**15.2**) | O(1) | O(1) |
| Balanced BST | O(log n) | O(log n) | O(log n) |
| **Heap** | **O(log n)** | **O(1)** | **O(log n)** |

A heap is the right shape for that question specifically. It is *worse* than a BST at everything else — which is the point.

### The heap property

> **Every parent is ≤ both of its children** (a min-heap).

```
            1
          /   \
        3       2          each parent <= its children
       / \     / \
      7   4   5   9        the MINIMUM is always at the root
```

🔴 **This is much weaker than a BST.** There is no left/right ordering — 3 sits left of 2, which a BST would forbid. A heap tells you the minimum and *nothing else*: searching for an arbitrary value is O(n), and an in-order traversal is meaningless.

That weakness is what makes it cheap to maintain.

## 🔴 A tree with no pointers

A heap is always a **complete** binary tree — every level full except possibly the last, which fills left to right. That regularity means it can be stored in a **flat array**, with the structure implied by arithmetic:

```
   index:    0    1    2    3    4    5    6
   array: [  1,   3,   2,   7,   4,   5,   9 ]

            1              for the node at index i:
          /   \
        3       2            parent      (i - 1) // 2
       / \     / \           left child   2i + 1
      7   4   5   9          right child  2i + 2
```

| Consequence | Why it matters |
|---|---|
| No node objects, no pointers | far less memory than **15.7**'s `TreeNode` |
| Contiguous memory | cache-friendly, unlike a pointer tree (**15.1**) |
| Structure is implicit | nothing to corrupt; no rebalancing bookkeeping |

This is why `heapq` operates on **an ordinary Python list**. There is no heap object — the list *is* the heap, and the invariant lives in the functions.

In [ ]:
def parent(i):
    return (i - 1) // 2


def left_child(i):
    return 2 * i + 1


def right_child(i):
    return 2 * i + 2


heap = [1, 3, 2, 7, 4, 5, 9]
print("array:", heap, "\n")
print(f"{'i':>3}{'value':>7}{'parent':>10}{'left':>10}{'right':>10}")
print("-" * 40)
for i in range(len(heap)):
    p = f"{heap[parent(i)]}" if i > 0 else "-"
    left = f"{heap[left_child(i)]}" if left_child(i) < len(heap) else "-"
    right = f"{heap[right_child(i)]}" if right_child(i) < len(heap) else "-"
    print(f"{i:>3}{heap[i]:>7}{p:>10}{left:>10}{right:>10}")

valid = all(heap[i] >= heap[parent(i)] for i in range(1, len(heap)))
print(f"\nheap property holds everywhere: {valid}")
print("minimum is heap[0]:", heap[0], " <- O(1), always")
print("\n🔴 But note the array is NOT sorted. A heap is not a sorted list;")
print("   it only guarantees the root. Everything else is partial order.")

### How push and pop keep the property

**Push — sift up.** Append to the end, then swap upward while it is smaller than its parent. At most one swap per level: **O(log n)**.

**Pop — sift down.** Take the root (the answer), move the *last* element into its place, then swap downward with the smaller child until the property holds again. Again one swap per level: **O(log n)**.

```
   pop from [1, 3, 2, 7, 4, 5, 9]

   1. answer = 1                       take the root
   2. move 9 to the root  [9, 3, 2, 7, 4, 5]
   3. 9 > min(3, 2) = 2   -> swap      [2, 3, 9, 7, 4, 5]
   4. 9 > 5               -> swap      [2, 3, 5, 7, 4, 9]
```

🔴 Step 2 is the part people get wrong: the *last* element goes to the root, not one of the children. That keeps the tree **complete**, which is what makes the array representation valid.

In [ ]:
def sift_up(heap, index):
    swaps = 0
    while index > 0 and heap[index] < heap[parent(index)]:
        heap[index], heap[parent(index)] = heap[parent(index)], heap[index]
        index = parent(index)
        swaps += 1
    return swaps


def sift_down(heap, index):
    size = len(heap)
    swaps = 0
    while True:
        smallest = index
        for child in (left_child(index), right_child(index)):
            if child < size and heap[child] < heap[smallest]:
                smallest = child
        if smallest == index:
            return swaps
        heap[index], heap[smallest] = heap[smallest], heap[index]
        index = smallest
        swaps += 1


def heap_push(heap, value):
    heap.append(value)
    return sift_up(heap, len(heap) - 1)


def heap_pop(heap):
    smallest = heap[0]
    last = heap.pop()                  # O(1) from the END (15.2)
    if heap:
        heap[0] = last                 # 🔴 the LAST element goes to the root
        sift_down(heap, 0)
    return smallest


heap = []
for value in (5, 3, 8, 1, 9, 2):
    swaps = heap_push(heap, value)
    print(f"  push {value} -> {str(heap):<24} ({swaps} swaps)")

print()
order = []
while heap:
    order.append(heap_pop(heap))
print("popping repeatedly gives sorted order:", order)
print("  ^ that is heap sort - O(n log n) - see 15.10")

## `heapq` - the real thing

The standard library implements all of the above in C, operating on a plain list.

| Function | Does | Cost |
|---|---|---|
| `heappush(h, x)` | insert | O(log n) |
| `heappop(h)` | remove and return the smallest | O(log n) |
| `h[0]` | peek at the smallest | **O(1)** |
| `heapify(h)` | turn a list into a heap **in place** | **O(n)** |
| `heappushpop(h, x)` | push then pop, in one step | O(log n), faster than both |
| `heapreplace(h, x)` | pop then push | O(log n) |
| `nlargest(k, it)` / `nsmallest(k, it)` | top k | O(n log k) |

### 🔴 `heapq` is min-only

There is no max-heap. The idiom is to **negate**:

```
    heapq.heappush(h, -value)      push the negative
    largest = -heapq.heappop(h)    negate again on the way out
```

For tuples, negate the sort key only: `(-priority, task)`. For objects that cannot be negated, wrap them or supply a sort key.

In [ ]:
import heapq

# ---- min-heap: the default ----
numbers = []
for value in (5, 3, 8, 1, 9, 2):
    heapq.heappush(numbers, value)
print("min-heap    :", numbers)
print("  peek      :", numbers[0], "  O(1)")
print("  pop       :", heapq.heappop(numbers))
print("  after pop :", numbers)

# ---- max-heap: negate ----
maximums = []
for value in (5, 3, 8, 1, 9, 2):
    heapq.heappush(maximums, -value)         # store the negative
print("\nmax-heap    :", [-v for v in maximums], " (stored as", maximums, ")")
print("  largest   :", -maximums[0])
print("  pop       :", -heapq.heappop(maximums))

# ---- heapify: an existing list, in place ----
existing = [9, 4, 7, 1, 8, 2, 6]
heapq.heapify(existing)                      # O(n), in place
print("\nheapify     :", existing)
print("  minimum   :", existing[0])

# ---- heappushpop: cheaper than push then pop ----
window = [3, 5, 8]
heapq.heapify(window)
evicted = heapq.heappushpop(window, 7)
print("\nheappushpop(7) on", [3, 5, 8], "-> evicted", evicted, "leaving", window)
print("  ^ one sift instead of two. This is the top-k workhorse.")

## 🔴 `heapify` is O(n), not O(n log n)

The obvious reasoning: *n* elements, each sifted into place at O(log n) — so O(n log n). **That is wrong**, and the reason is worth understanding.

`heapify` sifts **down**, starting from the last parent and working backwards. The cost of sifting a node down is bounded by its **height**, not the tree's:

```
   level          nodes      max sift distance     work
   ─────────────────────────────────────────────────────
   leaves          n/2              0               0        <- half the nodes!
   one above       n/4              1              n/4
   two above       n/8              2              2n/8
   ...
   root             1            log n           log n
```

Sum that series and it converges to **2n**. The many nodes are cheap; the expensive nodes are few.

> **Half the nodes in any binary tree are leaves**, and a leaf sifts down zero levels. That single fact is the whole proof.

### 🔴 And a correction to the usual comparison

The standard follow-up is *"so building by pushing n items is O(n log n)"*. That is the **worst case**, not what you will normally measure.

A push sifts **up**, and its cost is bounded by the tree's height — but a *random* value is usually already near the bottom and barely moves. Measured below, pushing n random values costs about **1.28n** swaps at every size: constant work per push.

To actually provoke O(n log n) you must feed it **descending** values, so every new item becomes the new minimum and sifts all the way to the root.

| Building a heap of n items | Random input | Descending input |
|---|---|---|
| `heapify` | O(n) | O(n) |
| n × `heappush` | O(n) in practice | **O(n log n)** |

> Use `heapify` when you already have the data — it is O(n) *regardless of input*, and immune to the adversarial case.

In [ ]:
def heapify_counted(data):
    """heapify by sifting DOWN from the last parent backwards. O(n)."""
    heap = list(data)
    swaps = 0
    for index in range(len(heap) // 2 - 1, -1, -1):    # last parent -> root
        swaps += sift_down(heap, index)
    return heap, swaps


def build_by_pushing(data):
    """Push one at a time. Each push sifts UP."""
    heap = []
    swaps = 0
    for value in data:
        swaps += heap_push(heap, value)
    return heap, swaps


import random

print(f"{'n':>8}{'input':>13}{'heapify':>12}{'push':>12}"
      f"{'heapify/n':>12}{'push/n':>10}")
print("-" * 68)
for n in (1_000, 4_000, 16_000, 64_000):
    rng = random.Random(15)
    inputs = (
        ("random", [rng.random() for _ in range(n)]),
        ("descending", list(range(n, 0, -1))),      # worst case for a min-heap
    )
    for label, data in inputs:
        _, heapify_swaps = heapify_counted(data)
        _, push_swaps = build_by_pushing(data)
        print(f"{n:>8}{label:>13}{heapify_swaps:>12,}{push_swaps:>12,}"
              f"{heapify_swaps / n:>12.2f}{push_swaps / n:>10.2f}")

print("\nRead the last two columns - work per element:")
print()
print("  heapify/n stays around 0.75-1.0 on BOTH inputs -> O(n), always.")
print()
print("  push/n on random input stays around 1.28 -> also O(n) in practice.")
print("  A random value is usually already near the bottom and barely moves.")
print()
print("  push/n on descending input rises 8 -> 10 -> 12 -> 14: it gains 2")
print("  per QUADRUPLING of n, which is 1 per doubling - that is log2(n).")
print("  Every value becomes the new minimum and sifts to the root.")
print()
print("🔴 So 'building by pushing is O(n log n)' is the WORST case, not the")
print("   typical one. 15.1 said to always state which case you mean - this")
print("   is why. heapify is O(n) either way, so prefer it when you can.")

check = [9, 4, 7, 1, 8, 2, 6]
mine, _ = heapify_counted(check)
theirs = list(check)
heapq.heapify(theirs)
print(f"\n  and it agrees with heapq.heapify: {mine == theirs}")

## Top-k - the heap's best-known trick

*"Find the 10 largest of a million values."*

| Approach | Cost | Memory |
|---|---|---|
| `sorted(data)[-10:]` | O(n log n) | O(n) |
| `heapq.nlargest(10, data)` | **O(n log k)** | **O(k)** |

**How it works:** keep a min-heap of size k. For each new value, if it beats the smallest of the k so far, replace it. The heap only ever holds k items, so each operation is O(log k) rather than O(log n).

With n = 1,000,000 and k = 10, log₂k is about 3 while log₂n is about 20 — nearly 7x less work, and 100,000x less memory.

> 🔴 **This is the answer to "top N" questions** — and it also works on a **stream**, where you cannot sort because you never hold the whole dataset.

In [ ]:
import time

rng = random.Random(15)
data = [rng.random() for _ in range(400_000)]
K = 10

started = time.perf_counter()
by_sorting = sorted(data, reverse=True)[:K]
sort_time = time.perf_counter() - started

started = time.perf_counter()
by_heap = heapq.nlargest(K, data)
heap_time = time.perf_counter() - started

print(f"top {K} of {len(data):,}\n")
print(f"  sorted(...)[:k]  {sort_time * 1000:8.1f} ms   O(n log n), O(n) memory")
print(f"  nlargest(k, ...) {heap_time * 1000:8.1f} ms   O(n log k), O(k) memory")
print(f"  ratio            {sort_time / heap_time:8.1f}x")
print(f"  same answer      {by_sorting == by_heap}")


def top_k_streaming(stream, k):
    """Works on data too large to hold. O(n log k) time, O(k) memory."""
    heap = []
    for value in stream:
        if len(heap) < k:
            heapq.heappush(heap, value)
        elif value > heap[0]:              # beats the weakest of the k
            heapq.heapreplace(heap, value)  # pop + push in one sift
    return sorted(heap, reverse=True)


print(f"\n  streaming version agrees: "
      f"{top_k_streaming(iter(data), K) == by_heap}")
print("  ^ and it never held more than", K, "items in memory.")

## Priority queues, and the tie-breaking trap

A heap of `(priority, item)` tuples is a priority queue. Tuples compare element-by-element, so the priority dominates.

🔴 **But when priorities tie, Python compares the second element** — and if that is an object with no ordering, you get `TypeError`. Worse, if it *is* comparable, you get silent ordering by payload, which is almost never what you meant.

```
    (2, {"task": "a"})  vs  (2, {"task": "b"})
     ^ tie                     -> compares the dicts -> TypeError
```

**The fix:** a monotonically increasing counter as the middle element.

```
    (priority, next(counter), item)
               ^^^^^^^^^^^^^ unique, so the item is never compared
```

This also makes the queue **stable**: equal priorities come out in insertion order — usually the behaviour you want, and rarely the behaviour you get by accident.

In [ ]:
import itertools

# ---- 🔴 the trap ----
queue = []
heapq.heappush(queue, (2, {"task": "deploy"}))
try:
    heapq.heappush(queue, (2, {"task": "rollback"}))     # same priority
except TypeError as exc:
    print("pushing a tied priority with a dict payload:")
    print("  TypeError:", exc)
    print("  ^ the tie made Python compare the DICTS\n")


# ---- ✅ the fix ----
class PriorityQueue:
    """Stable, and safe with any payload."""

    def __init__(self):
        self._heap = []
        self._counter = itertools.count()

    def push(self, item, priority):
        # the counter is unique, so `item` is never reached by a comparison
        heapq.heappush(self._heap, (priority, next(self._counter), item))

    def pop(self):
        if not self._heap:
            raise IndexError("pop from an empty priority queue")
        priority, _order, item = heapq.heappop(self._heap)
        return item, priority

    def __len__(self):
        return len(self._heap)


jobs = PriorityQueue()
for task, priority in (("send-digest", 3), ("page-oncall", 1), ("reindex", 2),
                      ("purge-cache", 3), ("restart-worker", 1)):
    jobs.push({"task": task}, priority)

print("draining by priority:")
while len(jobs):
    item, priority = jobs.pop()
    print(f"  priority {priority}: {item['task']}")

print("\n  Dict payloads, and no TypeError.")
print("  page-oncall before restart-worker, and send-digest before")
print("  purge-cache: ties broke by INSERTION ORDER, which is stability.")

## Merging k sorted sequences

The follow-up to merging two lists (**15.4**). With k sequences totalling N items:

| Approach | Cost |
|---|---|
| Concatenate and sort | O(N log N) — throws away the existing order |
| Merge one at a time | O(N·k) — the first list is re-walked every time |
| **Heap of the k current heads** | **O(N log k)** |

The heap holds at most k items — one per sequence — so each of the N extractions costs O(log k).

`heapq.merge()` does exactly this, lazily, and works on **iterators** — so it can merge sorted files far larger than memory. That is the external merge sort used by databases (**10.1**).

In [ ]:
def merge_k_sorted(sequences):
    """Heap of (value, which sequence, position). O(N log k)."""
    heap = []
    for index, sequence in enumerate(sequences):
        if sequence:
            heapq.heappush(heap, (sequence[0], index, 0))

    out = []
    while heap:
        value, which, position = heapq.heappop(heap)
        out.append(value)
        following = position + 1
        if following < len(sequences[which]):
            heapq.heappush(heap, (sequences[which][following], which, following))
    return out


lists = [[1, 5, 9], [2, 6, 10], [3, 7, 11], [4, 8, 12]]
print("input   :", lists)
print("merged  :", merge_k_sorted(lists))
print("correct :", merge_k_sorted(lists) == sorted(sum(lists, [])))

print("\nthe standard library does it lazily:")
print("  heapq.merge ->", list(heapq.merge(*lists)))


def counting_generator(start, step, count, log):
    """Records how many items are actually pulled."""
    for i in range(count):
        log.append(start + i * step)
        yield start + i * step


pulled = []
streams = [counting_generator(s, 4, 1_000, pulled) for s in range(4)]
first_five = list(itertools.islice(heapq.merge(*streams), 5))
print(f"\n  first 5 of a merged 4,000-item stream: {first_five}")
print(f"  items actually generated: {len(pulled)} of 4,000")
print("  ^ laziness: it merged only what was asked for. That is how you")
print("    merge sorted files bigger than memory.")

## Running median with two heaps

*"Report the median after every new number in a stream."* Sorting each time is O(n log n) per query. Two heaps make it O(log n) per insert and **O(1)** per query.

```
        lower half              upper half
     MAX-heap (negated)        MIN-heap
     [ ... , 3 ]  <- largest   smallest -> [ 5, ... ]
               \                        /
                the median lives between them
```

**The invariants:**
1. every value in `lower` ≤ every value in `upper`
2. their sizes differ by at most one

Then the median is the top of the larger heap, or the average of both tops when the sizes are equal.

🔴 The subtlety: after pushing, a value may be on the wrong side. **Push to one heap, immediately move its top to the other**, then rebalance — that guarantees invariant 1 without any comparison logic.

In [ ]:
class RunningMedian:
    """O(log n) per add, O(1) per median query."""

    def __init__(self):
        self._lower = []      # max-heap, stored negated
        self._upper = []      # min-heap

    def add(self, value):
        # Push to lower, then hand its largest to upper. This guarantees
        # everything in lower <= everything in upper, with no branching.
        heapq.heappush(self._lower, -value)
        heapq.heappush(self._upper, -heapq.heappop(self._lower))
        # rebalance so lower is never smaller than upper
        if len(self._upper) > len(self._lower):
            heapq.heappush(self._lower, -heapq.heappop(self._upper))

    def median(self):
        if not self._lower:
            raise ValueError("no values yet")
        if len(self._lower) > len(self._upper):
            return float(-self._lower[0])
        return (-self._lower[0] + self._upper[0]) / 2


import statistics

tracker = RunningMedian()
seen = []
for value in (5, 15, 1, 3, 8, 7, 9, 2):
    tracker.add(value)
    seen.append(value)
    mine = tracker.median()
    theirs = statistics.median(seen)
    flag = "ok" if abs(mine - theirs) < 1e-9 else "MISMATCH"
    print(f"  add {value:>2} -> median {mine:>5.1f}   "
          f"(statistics.median: {theirs:>5.1f}) {flag}")

rng = random.Random(15)
tracker = RunningMedian()
values = []
agree = True
for _ in range(500):
    value = rng.randint(0, 1_000)
    tracker.add(value)
    values.append(value)
    if abs(tracker.median() - statistics.median(values)) > 1e-9:
        agree = False
        break
print(f"\n  agrees with statistics.median across 500 random inserts: {agree}")

## Interview questions

**1. What is a heap, and how does it differ from a BST?**
> A complete binary tree with the heap property, stored in an array. It orders only parent-to-child, so it gives O(1) access to the minimum but O(n) search for anything else. A BST orders left-to-right and gives O(log n) search.

**2. Why is `heapify` O(n) rather than O(n log n)?**
> It sifts **down**, and the cost per node is its height. Half the nodes are leaves with height 0; the series sums to 2n.

**3. Kth largest element in an array.**
> A min-heap of size k: O(n log k). Or quickselect for O(n) average (**15.11**). Sorting is O(n log n) and is the answer to improve on.

**4. Top k frequent elements.**
> `Counter` (**15.6**) then `nlargest(k, ...)`: O(n log k). Bucket sort by frequency gives O(n) if you want to push further.

**5. Merge k sorted lists.** *(above)*
> Heap of the k heads: O(N log k). Mention that `heapq.merge` is lazy and does this already.

**6. Find the median of a data stream.** *(above)*
> Two heaps. O(log n) insert, O(1) query.

**7. How would you implement a max-heap with `heapq`?**
> Negate on the way in and out. For tuples, negate only the sort key.

**8. Task scheduler / meeting rooms.**
> Sort by start time, then a min-heap of end times. The heap size is the number of concurrent rooms needed.

**9. Why does a heap use an array rather than nodes?**
> It is always complete, so positions are computable — `2i+1`, `2i+2`. That saves the pointers and gives contiguous, cache-friendly memory.

**10. Can you delete an arbitrary element from a heap?**
> Not efficiently — finding it is O(n). The standard workaround is **lazy deletion**: mark it deleted in a set and discard it when it surfaces at the top. That is how Dijkstra is usually implemented (**15.9**).

In [ ]:
# Questions 4, 8 and 10 - the three that come up most.
from collections import Counter


def top_k_frequent(items, k):
    counts = Counter(items)                       # O(n)
    return [item for item, _ in counts.most_common(k)]


words = "deploy build deploy test build deploy release test deploy build".split()
print("top 2 frequent:", top_k_frequent(words, 2))
print("  via nlargest:", heapq.nlargest(2, Counter(words).items(),
                                        key=lambda pair: pair[1]))


def min_meeting_rooms(intervals):
    """Fewest rooms needed. Sort by start, min-heap of end times."""
    if not intervals:
        return 0
    ends = []
    for start, end in sorted(intervals):          # by start time
        if ends and ends[0] <= start:
            heapq.heappop(ends)                   # a room freed up
        heapq.heappush(ends, end)
    return len(ends)                              # peak concurrency


print()
for meetings in ([(0, 30), (5, 10), (15, 20)],
                [(7, 10), (2, 4)],
                [(1, 5), (2, 6), (3, 7)],
                []):
    print(f"  {str(meetings):<34} needs {min_meeting_rooms(meetings)} room(s)")


class LazyDeleteHeap:
    """Arbitrary deletion, the standard way: mark and discard on surfacing."""

    def __init__(self):
        self._heap = []
        self._removed = Counter()

    def push(self, value):
        heapq.heappush(self._heap, value)

    def remove(self, value):
        self._removed[value] += 1        # O(1) - do not touch the heap

    def pop(self):
        while self._heap:
            value = heapq.heappop(self._heap)
            if self._removed[value]:
                self._removed[value] -= 1
                continue                 # it was deleted; skip it
            return value
        raise IndexError("pop from an empty heap")


lazy = LazyDeleteHeap()
for value in (5, 1, 8, 3, 9):
    lazy.push(value)
lazy.remove(1)
lazy.remove(8)
print("\n  after removing 1 and 8, popping gives:",
      [lazy.pop() for _ in range(3)])
print("  ^ deletion was O(1); the cost is deferred to the pops, and the")
print("    heap may hold stale entries in the meantime.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Expecting a heap to be sorted.** Only the root is guaranteed. Searching a heap is O(n).
2. 🔴 **Forgetting `heapq` is min-only.** Negate for a max-heap - and negate only the sort key in a tuple.
3. 🔴 **Pushing tuples whose payload is not comparable.** A tie sends Python to the next element. Add a counter.
4. 🔴 **Assuming `heapify` is O(n log n).** It is O(n) - sifting down, with half the nodes costing nothing.
5. **Sorting to get the top k.** O(n log n) where `nlargest` is O(n log k) and uses O(k) memory.
6. **Moving a child to the root in pop.** It must be the *last* element, or the tree stops being complete.
7. **Trying to delete an arbitrary element.** Finding it is O(n); use lazy deletion.
8. **Mutating a value already in the heap.** The invariant breaks silently; remove and re-push instead.
9. **Using a heap when you need ordering.** Use a sorted structure or a BST (**15.7**).

## Best Practices

- Use `heapq` rather than writing your own; it is C and correct.
- Use `heapify` when you already have all the data - O(n) beats n pushes.
- Use `nlargest`/`nsmallest` for top-k, and a size-k heap for streams.
- Use `heappushpop`/`heapreplace` when you push and pop together - one sift, not two.
- Put a unique counter in priority tuples for safety and stability.
- Reach for two heaps whenever a problem needs a running middle.
- Use lazy deletion rather than searching the heap.
- State whether you need the min, the max, or both - it determines the structure.

## Practice Exercises

Try these before moving on.

1. Implement a max-heap class wrapping `heapq` with the negation hidden inside, so callers never see it.
2. 🔴 Extend the `heapify` counting cell to n = 256,000 and confirm the swap count stays below n while the push count keeps growing faster.
3. Implement `kth_largest` three ways - sorting, a size-k heap, and `nlargest` - and compare timings for n = 1,000,000, k = 5.
4. Add `peek()` and `__contains__` to `LazyDeleteHeap`. Why is `__contains__` awkward, and what would it cost?
5. Implement a task scheduler where each task has a priority *and* a deadline. Which should be the primary sort key, and why does the order matter?
6. Use `heapq.merge` to merge three sorted files on disk without loading them into memory (**08 File Handling**).
7. Extend `RunningMedian` to support removing a value. What breaks, and how does lazy deletion help?
8. 🔴 Sort a list by repeatedly calling `heappop` and confirm the result. What is the complexity, and how does it compare with `sorted()` (**15.10**)?